In [ ]:
!pip install -q torch transformers datasets matplotlib

In [ ]:
import re
import time

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)
MODEL_NAME = "Qwen/Qwen3.5-0.8B"
JUDGE_NAME = "Qwen/Qwen3Guard-Gen-0.6B"
SEED = 0

torch.manual_seed(SEED)
transformers.logging.set_verbosity_error()
print(f"Using device: {DEVICE}")

# Representation Engineering

---
# 1) Introduction

A transformer's residual stream $h_\ell \in \mathbb{R}^{d}$ (the running sum every layer reads from
and writes to) encodes many high-level properties of the input as **linear directions**: whether
a request is harmful, the sentiment of a review, the language of the text. If that holds, a model's
behaviour can be read and edited with linear algebra at inference time, without gradient-based
fine-tuning.

**Representation engineering** is the toolkit built on this:

| Tool | What it does | Needs labels? |
|---|---|---|
| Difference-in-means direction | One vector per concept from two contrastive prompt sets | Yes, two prompt sets |
| Linear probe | Reads the concept off $h_\ell$; shows *where* it becomes linearly decodable | Yes |
| Activation addition (steering) | $h \leftarrow h + \alpha\,\hat r$: push the model towards a behaviour | Direction only |
| Directional ablation | $h \leftarrow h - (h^\top \hat r)\hat r$: remove a behaviour | Direction only |
| Weight orthogonalisation ("abliteration") | Bake the ablation into the weights; no hooks at inference | Direction only |
| Sparse autoencoder (SAE) | Unsupervised dictionary of thousands of features; find directions nobody labelled | No |

Refusal is the running example because it is a clean, behaviourally measurable concept: Arditi et
al. (2024) showed that refusal in chat models is mediated by a single direction. The same code
works for any concept with two contrastive prompt sets.

> Refusal removal is studied here because it shows how fragile safety fine-tuning is to a rank-1
> edit. That fragility is the main reason open-weight safety cannot rely on refusal training
> alone. Generations from the ablated model are truncated in this notebook.

---
# 2) Setup

- **Model.** `Qwen3.5-0.8B` instruct, 24 layers, `embed_dim` 1024, hybrid linear/full attention.
- **Data.** `mlabonne/harmful_behaviors` (AdvBench-style harmful instructions) and
  `mlabonne/harmless_alpaca`. Directions are fitted on the train split, and every number is
  reported on held-out test prompts.
- **Judge.** Refusal is scored by `Qwen3Guard-Gen-0.6B` in response-moderation mode, which emits
  `Refusal: Yes/No` for a (prompt, response) pair. A keyword list of "I'm sorry" phrases misses
  soft refusals and flags compliant answers that mention safety, so a model makes this call.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, padding_side="left")
model = (
    AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.bfloat16)
    .to(DEVICE)
    .eval()
)
layers = model.model.layers
NUM_LAYERS, EMBED_DIM = len(layers), model.config.get_text_config().hidden_size
print(f"{MODEL_NAME}: {NUM_LAYERS} layers, embed_dim {EMBED_DIM}")

harmful = load_dataset("mlabonne/harmful_behaviors")
harmless = load_dataset("mlabonne/harmless_alpaca")
NUM_FIT, NUM_TEST = 96, 32
harmful_train, harmful_held = list(harmful["train"]["text"]), list(
    harmful["test"]["text"]
)
harmful_fit = harmful_train[:NUM_FIT]
harmless_fit = list(harmless["train"]["text"][:NUM_FIT])
harmful_test = harmful_held[:NUM_TEST]
harmless_test = list(harmless["test"]["text"][:NUM_TEST])
print(
    f"Fit: {len(harmful_fit)} + {len(harmless_fit)}, test: {len(harmful_test)} + {len(harmless_test)}"
)
print(f"Harmful example:  {harmful_fit[0]}")
print(f"Harmless example: {harmless_fit[0]}")

In [ ]:
def chat(prompts: list[str]) -> list[str]:
    """Apply the model's chat template (non-thinking) to user prompts."""
    return [
        tokenizer.apply_chat_template(
            [{"role": "user", "content": p}], tokenize=False, add_generation_prompt=True
        )
        for p in prompts
    ]


@torch.no_grad()
def generate(
    prompts: list[str], max_new_tokens: int = 64, chunk: int = 16
) -> list[str]:
    """Greedy completions for a list of user prompts."""
    out = []
    for start in range(0, len(prompts), chunk):
        batch = tokenizer(
            chat(prompts[start : start + chunk]), return_tensors="pt", padding=True
        ).to(DEVICE)
        ids = model.generate(**batch, max_new_tokens=max_new_tokens, do_sample=False)
        out += tokenizer.batch_decode(
            ids[:, batch["input_ids"].size(1) :], skip_special_tokens=True
        )
    return out


judge_tokenizer = AutoTokenizer.from_pretrained(JUDGE_NAME, padding_side="left")
judge = (
    AutoModelForCausalLM.from_pretrained(JUDGE_NAME, dtype=torch.bfloat16)
    .to(DEVICE)
    .eval()
)


@torch.no_grad()
def refusal_rate(prompts: list[str], responses: list[str], chunk: int = 16) -> float:
    """Fraction of responses the guard model labels 'Refusal: Yes'."""
    labels = []
    for start in range(0, len(prompts), chunk):
        texts = [
            judge_tokenizer.apply_chat_template(
                [{"role": "user", "content": p}, {"role": "assistant", "content": r}],
                tokenize=False,
            )
            for p, r in zip(
                prompts[start : start + chunk], responses[start : start + chunk]
            )
        ]
        batch = judge_tokenizer(texts, return_tensors="pt", padding=True).to(DEVICE)
        ids = judge.generate(**batch, max_new_tokens=32, do_sample=False)
        for text in judge_tokenizer.batch_decode(
            ids[:, batch["input_ids"].size(1) :], skip_special_tokens=True
        ):
            match = re.search(r"Refusal: (Yes|No)", text)
            labels.append(match is not None and match.group(1) == "Yes")
    return sum(labels) / len(labels)

In [ ]:
start = time.time()
base_harmful = generate(harmful_test)
base_harmless = generate(harmless_test)
print(
    f"Baseline refusal rate, harmful test:  {refusal_rate(harmful_test, base_harmful):.0%}"
)
print(
    f"Baseline refusal rate, harmless test: {refusal_rate(harmless_test, base_harmless):.0%}"
)
print(f"({time.time() - start:.0f}s for {2 * NUM_TEST} generations + judging)")
print(f"\n{harmful_test[0]}\n -> {base_harmful[0][:150]!r}")

---
# 3) The math

Let $a_\ell(x) \in \mathbb{R}^{d}$ be the residual stream after layer $\ell$ at the **last prompt
token** (the position that produces the first response token). For two prompt sets
$\mathcal{D}_{+}$ (harmful) and $\mathcal{D}_{-}$ (harmless):

**Difference in means.**
$$
r_\ell = \frac{1}{|\mathcal{D}_+|}\sum_{x\in\mathcal{D}_+} a_\ell(x) - \frac{1}{|\mathcal{D}_-|}\sum_{x\in\mathcal{D}_-} a_\ell(x),
\qquad \hat r_\ell = r_\ell / \lVert r_\ell \rVert .
$$
This is the optimal linear discriminant when both classes share an isotropic covariance, and it
is far more robust than a trained probe with only ~100 examples per class.

**Activation addition.** At every position of a chosen layer, $h \leftarrow h + \alpha\,\hat r$.
Positive $\alpha$ pushes towards the "+" behaviour.

**Directional ablation.** Every write into the residual stream (the embedding, each attention or
linear-attention output, each MLP output) is projected, $o \leftarrow (I - \hat r \hat r^\top) o$, so the
stream never carries a component along $\hat r$ and no layer can read it.

**Weight orthogonalisation.** Every matrix $W_{\text{out}} \in \mathbb{R}^{d \times k}$ that
*writes* into the residual stream (token embedding, attention output, linear-attention output,
MLP down-projection) is replaced by
$$
W_{\text{out}}' = (I - \hat r \hat r^\top)\, W_{\text{out}} .
$$
Since the residual stream is a sum of these writes, no component can ever put anything along
$\hat r$; this is exactly directional ablation, with zero inference overhead.

**Sparse autoencoder (TopK).** With pre-bias $b_{\text{dec}} \in \mathbb{R}^{d}$, encoder
$W_{\text{enc}} \in \mathbb{R}^{m \times d}$, decoder $W_{\text{dec}} \in \mathbb{R}^{d \times m}$ (unit-norm
columns) and dictionary size $m \gg d$:
$$
z = \operatorname{TopK}\big(W_{\text{enc}}(a - b_{\text{dec}}) + b_{\text{enc}}\big),\qquad
\hat a = W_{\text{dec}}\, z + b_{\text{dec}},\qquad
\mathcal{L} = \lVert a - \hat a\rVert_2^2 ,
$$
where TopK keeps the $k$ largest pre-activations and zeroes the rest. Each column of
$W_{\text{dec}}$ is a candidate feature direction; TopK fixes the sparsity $L_0 = k$ exactly instead
of tuning an $\ell_1$ penalty (Gao et al., 2024).

---
# 4) From-scratch implementation

## 4.1 Capturing the residual stream

In [ ]:
@torch.no_grad()
def last_token_acts(prompts: list[str], chunk: int = 16) -> torch.Tensor:
    """Residual stream after every layer at the last prompt token.

    Returns (num_prompts, num_layers, embed_dim) on CPU.
    """
    out = []
    for start in range(0, len(prompts), chunk):
        batch = tokenizer(
            chat(prompts[start : start + chunk]), return_tensors="pt", padding=True
        ).to(DEVICE)
        hidden = model(**batch, output_hidden_states=True).hidden_states
        # hidden_states[0] is the embedding output; [1:] are the layer outputs
        # num_layers x (batch_num, seq_len, embed_dim) -> (batch_num, num_layers, embed_dim)
        out.append(torch.stack([h[:, -1] for h in hidden[1:]], dim=1).float().cpu())
    return torch.cat(out)


acts_pos, acts_neg = last_token_acts(harmful_fit), last_token_acts(harmless_fit)
test_pos, test_neg = last_token_acts(harmful_test), last_token_acts(harmless_test)
print(f"Fit activations: {tuple(acts_pos.shape)} and {tuple(acts_neg.shape)}")

## 4.2 Difference-in-means directions and where the concept lives

Each layer's direction is fitted on the train split. The held-out accuracy of the simplest
possible probe, thresholding $a^\top \hat r_\ell$ at the midpoint of the two class means, shows at
which depth "harmful request" becomes linearly readable.

In [ ]:
def diff_in_means(pos: torch.Tensor, neg: torch.Tensor) -> torch.Tensor:
    """Per-layer unit directions; pos, neg (num_prompts, num_layers, embed_dim)."""
    return F.normalize(pos.mean(0) - neg.mean(0), dim=-1)


def probe_accuracy(
    direction: torch.Tensor, pos: torch.Tensor, neg: torch.Tensor, ref_pos, ref_neg
) -> torch.Tensor:
    """Held-out accuracy per layer of the midpoint threshold on a^T r."""
    # (num_prompts, num_layers)
    proj = lambda a: (a * direction).sum(-1)
    threshold = (proj(ref_pos).mean(0) + proj(ref_neg).mean(0)) / 2
    correct = (proj(pos) > threshold).float().sum(0) + (
        proj(neg) <= threshold
    ).float().sum(0)
    return correct / (len(pos) + len(neg))


directions = diff_in_means(acts_pos, acts_neg)
accuracy = probe_accuracy(directions, test_pos, test_neg, acts_pos, acts_neg)
plt.figure(figsize=(7, 3))
plt.plot(range(NUM_LAYERS), accuracy, marker="o")
plt.xlabel("layer")
plt.ylabel("held-out accuracy")
plt.title("Linear readability of 'harmful request'")
plt.tight_layout()
plt.show()
print(f"Accuracy by layer: {[round(a, 2) for a in accuracy.tolist()]}")

Readability alone does not pick the direction to intervene with: early layers may separate the two
sets by surface features (topic words) rather than by the model's decision to refuse. The
standard selection is **causal**: for candidate layers in the middle of the network, ablate each
layer's direction everywhere and keep the one that lowers refusal most on a small validation set
of harmful prompts, while keeping the harmless next-token distribution close to the original.

## 4.3 Directional ablation with hooks

In [ ]:
def residual_writers(model: nn.Module) -> list[nn.Module]:
    """Every module whose output is added into the residual stream."""
    writers = [model.model.embed_tokens]
    for layer in model.model.layers:
        writers.append(
            layer.self_attn.o_proj
            if hasattr(layer, "self_attn")
            else layer.linear_attn.out_proj
        )
        writers.append(layer.mlp.down_proj)
    return writers


class ResidualHook:
    """Ablate a direction from every residual write, or add it to chosen layer outputs.

    Ablation hooks the writers (embedding, attention / linear-attention output, MLP output), so
    the residual stream never contains the direction at any point, including mid-layer.
    """

    def __init__(
        self,
        model: nn.Module,
        direction: torch.Tensor,
        mode: str,
        alpha: float = 0.0,
        layer_ids=None,
    ):
        self.direction = direction.to(DEVICE, torch.float32)
        self.mode, self.alpha = mode, alpha
        if mode == "ablate":
            targets = residual_writers(model)
        else:
            ids = range(len(model.model.layers)) if layer_ids is None else layer_ids
            targets = [model.model.layers[i] for i in ids]
        self.handles = [m.register_forward_hook(self.hook) for m in targets]

    def hook(self, module: nn.Module, args, output):
        h = output[0] if isinstance(output, tuple) else output
        r = self.direction.to(h.dtype)
        if self.mode == "ablate":
            # (batch_num, seq_len, embed_dim) - (batch_num, seq_len, 1) * (embed_dim,)
            h = h - (h @ r).unsqueeze(-1) * r
        else:
            h = h + self.alpha * r
        return (h,) + tuple(output[1:]) if isinstance(output, tuple) else h

    def remove(self) -> None:
        for handle in self.handles:
            handle.remove()


@torch.no_grad()
def next_token_kl(prompts: list[str], reference_logps: torch.Tensor) -> float:
    """Mean KL(reference || current) of the first response token."""
    batch = tokenizer(chat(prompts), return_tensors="pt", padding=True).to(DEVICE)
    logps = model(**batch).logits[:, -1].float().log_softmax(-1).cpu()
    return F.kl_div(
        logps, reference_logps, log_target=True, reduction="batchmean"
    ).item()


with torch.no_grad():
    batch = tokenizer(chat(harmless_test), return_tensors="pt", padding=True).to(DEVICE)
    harmless_ref_logps = model(**batch).logits[:, -1].float().log_softmax(-1).cpu()

In [ ]:
val_prompts = harmful_train[NUM_FIT : NUM_FIT + 16]
candidates = list(range(NUM_LAYERS // 3, 3 * NUM_LAYERS // 4, 2))
scores = {}
for layer in candidates:
    hook = ResidualHook(model, directions[layer], "ablate")
    rate = refusal_rate(val_prompts, generate(val_prompts, max_new_tokens=32))
    kl = next_token_kl(harmless_test, harmless_ref_logps)
    hook.remove()
    scores[layer] = (rate, kl)
    print(
        f"layer {layer:>2}: refusal on val harmful {rate:.0%}, harmless first-token KL {kl:.3f}"
    )

BEST_LAYER = min(scores, key=lambda l: (scores[l][0], scores[l][1]))
refusal_dir = directions[BEST_LAYER]
print(f"Selected layer {BEST_LAYER}")

In [ ]:
hook = ResidualHook(model, refusal_dir, "ablate")
ablated_harmful = generate(harmful_test)
ablated_harmless = generate(harmless_test)
ablated_kl = next_token_kl(harmless_test, harmless_ref_logps)
hook.remove()
print(
    f"Ablated refusal rate, harmful test:  {refusal_rate(harmful_test, ablated_harmful):.0%}"
)
print(
    f"Ablated refusal rate, harmless test: {refusal_rate(harmless_test, ablated_harmless):.0%}"
)
print(f"Harmless first-token KL vs original: {ablated_kl:.3f}")
print(f"\n{harmful_test[0]}\n -> {ablated_harmful[0][:100]!r}...")

## 4.4 Weight orthogonalisation (abliteration)

The residual stream is a sum of writes, so projecting $\hat r$ out of every writing matrix removes it
at every layer output. In this hybrid model the writers are the token embedding, the attention
`o_proj` (full-attention layers), the Gated DeltaNet `out_proj` (linear-attention layers) and the
MLP `down_proj`. A writer $W \in \mathbb{R}^{d\times k}$ in `nn.Linear` storage has shape
`(out_features, in_features) = (embed_dim, k)`, so the projection multiplies from the left; the
embedding stores `(vocab_size, embed_dim)` and projects from the right.

In [ ]:
@torch.no_grad()
def orthogonalize_weights(model: nn.Module, direction: torch.Tensor) -> int:
    """W <- (I - r r^T) W for every matrix that writes into the residual stream."""
    r = direction.to(DEVICE, torch.float32)
    writers = residual_writers(model)
    # (vocab_size, embed_dim) - (vocab_size, 1) * (1, embed_dim)
    emb = writers[0].weight.float()
    writers[0].weight.copy_(emb - (emb @ r).unsqueeze(1) * r)
    for linear in writers[1:]:
        # (embed_dim, k) - (embed_dim, 1) * (1, k)
        w = linear.weight.float()
        linear.weight.copy_(w - r.unsqueeze(1) * (r @ w).unsqueeze(0))
    return len(writers)


original_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
edited = orthogonalize_weights(model, refusal_dir)
print(f"Orthogonalised {edited} matrices")

The check: logits from the edited weights must match logits from the hooked original model.
Both remove $\hat r$ from exactly the same writes (the hooks sit on the writer modules, the edit
on their weights), so the two models compute the same function. What remains is bf16 rounding:
the edited weights are rounded once, the hooked activations at every step, which can add up to a
few tenths of a logit after 24 layers without changing the predicted token.

In [ ]:
with torch.no_grad():
    probe_batch = tokenizer(
        chat(harmful_test[:8]), return_tensors="pt", padding=True
    ).to(DEVICE)
    logits_weights = model(**probe_batch).logits[:, -1].float()
    edited_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(original_state)
    hook = ResidualHook(model, refusal_dir, "ablate")
    logits_hooks = model(**probe_batch).logits[:, -1].float()
    hook.remove()
    model.load_state_dict(edited_state)

diff = (logits_weights - logits_hooks).abs().max().item()
agree = (logits_weights.argmax(-1) == logits_hooks.argmax(-1)).float().mean().item()
print(f"Max |logit difference|: {diff:.2e}, argmax agreement {agree:.0%}")
abliterated_harmful = generate(harmful_test)
print(
    f"Abliterated weights, refusal on harmful test: {refusal_rate(harmful_test, abliterated_harmful):.0%}"
)
model.load_state_dict(original_state)

## 4.5 Activation addition: inducing refusal

The same direction, added instead of removed, at the selected layer only, makes the model
refuse harmless requests. The sweep over $\alpha$ is in units of the typical projection gap
$\lVert r_\ell \rVert$ between the two classes at that layer.
Responses are cut to 40 tokens to keep the sweep fast, so the judge flags a few truncated
harmless answers even at $\alpha = 0$; compare each row against that row, not the full-length baseline.

In [ ]:
gap = (acts_pos[:, BEST_LAYER].mean(0) - acts_neg[:, BEST_LAYER].mean(0)).norm().item()
print(f"Class-mean gap at layer {BEST_LAYER}: {gap:.1f}")
for scale in [0.0, 0.5, 1.0, 1.5]:
    hook = ResidualHook(
        model, refusal_dir, "add", alpha=scale * gap, layer_ids=[BEST_LAYER]
    )
    responses = generate(harmless_test[:16], max_new_tokens=40)
    hook.remove()
    print(
        f"alpha = {scale:.1f} x gap: refusal on harmless {refusal_rate(harmless_test[:16], responses):.0%}"
        f" | {harmless_test[0][:40]!r} -> {responses[0][:70]!r}"
    )

## 4.6 A sparse autoencoder on the residual stream

The direction above needed two labelled prompt sets. A sparse autoencoder finds directions
**without labels**: it learns an overcomplete dictionary ($m = 8d = 8192$ features) such that each
activation is reconstructed from only $k = 32$ of them. It is trained on residual activations at
`BEST_LAYER` over ordinary web text (FineWeb-Edu) and chat-formatted prompts, all token positions.
Here it sees 250K tokens; production SAEs (Gemma Scope, Llama Scope) use the same objective on
billions of tokens with $m$ up to $10^6$.

In [ ]:
SAE_TOKENS = 250_000
CTX = 256


@torch.no_grad()
def collect_layer_acts(texts: list[str], layer: int, budget: int) -> torch.Tensor:
    """Residual activations at one layer for all non-pad tokens: (num_tokens, embed_dim)."""
    store, total = [], 0
    for start in range(0, len(texts), 8):
        batch = tokenizer(
            texts[start : start + 8],
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=CTX,
        ).to(DEVICE)
        hidden = model(**batch, output_hidden_states=True).hidden_states[layer + 1]
        # drop pads and the first position (its norm is an outlier, as in most LLMs)
        mask = batch["attention_mask"].bool()
        mask[:, :1] = False
        mask &= torch.arange(mask.size(1), device=DEVICE) >= (
            mask.size(1) - batch["attention_mask"].sum(1, keepdim=True) + 1
        )
        store.append(hidden[mask].half().cpu())
        total += store[-1].size(0)
        if total >= budget:
            break
    return torch.cat(store)[:budget]


web = [
    d["text"]
    for d, _ in zip(
        load_dataset(
            "HuggingFaceFW/fineweb-edu", "sample-10BT", split="train", streaming=True
        ),
        range(1600),
    )
]
chat_texts = chat(harmful_train + list(harmless["train"]["text"][:1000]))
start = time.time()
sae_acts = torch.cat(
    [
        collect_layer_acts(web, BEST_LAYER, int(SAE_TOKENS * 0.8)),
        collect_layer_acts(chat_texts, BEST_LAYER, int(SAE_TOKENS * 0.2)),
    ]
)
sae_acts = sae_acts[torch.randperm(len(sae_acts))]
print(
    f"Collected {tuple(sae_acts.shape)} activations in {time.time() - start:.0f}s "
    f"({sae_acts.numel() * 2 / 2**30:.2f} GiB fp16)"
)

In [ ]:
class TopKSAE(nn.Module):
    """TopK sparse autoencoder with unit-norm decoder columns."""

    def __init__(self, embed_dim: int, num_latents: int, k: int) -> None:
        super().__init__()
        self.k = k
        self.b_dec = nn.Parameter(torch.zeros(embed_dim))
        self.encoder = nn.Linear(embed_dim, num_latents)
        # (embed_dim, num_latents): initialised as the encoder's transpose
        self.W_dec = nn.Parameter(self.encoder.weight.detach().T.clone())
        self.normalize_decoder()

    @torch.no_grad()
    def normalize_decoder(self) -> None:
        self.W_dec.div_(self.W_dec.norm(dim=0, keepdim=True))

    def encode(self, a: torch.Tensor) -> torch.Tensor:
        """(batch_num, embed_dim) -> sparse codes (batch_num, num_latents)."""
        pre = F.relu(self.encoder(a - self.b_dec))
        top = pre.topk(self.k, dim=-1)
        return torch.zeros_like(pre).scatter_(-1, top.indices, top.values)

    def forward(self, a: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        z = self.encode(a)
        return z @ self.W_dec.T + self.b_dec, z


sae = TopKSAE(EMBED_DIM, 8 * EMBED_DIM, k=32).to(DEVICE)
probe_acts = sae_acts[:4].float().to(DEVICE)
recon, codes = sae(probe_acts)
print(
    f"Input {tuple(probe_acts.shape)} -> codes {tuple(codes.shape)}, L0 = {(codes > 0).sum(-1).tolist()}"
)

Training details that matter: activations are scaled by one constant so their mean squared
norm is $d$ (so the learning rate does not depend on the layer's norm); $b_{\text{dec}}$ starts at the
data mean; decoder columns are renormalised after every step, and the gradient component parallel
to each column is removed first so the step is not wasted on changing the norm. The quality metric
is the **fraction of variance unexplained**,
$\text{FVU} = \lVert a - \hat a\rVert^2 / \lVert a - \bar a\rVert^2$.

In [ ]:
SCALE = (sae_acts[:50_000].float().pow(2).sum(-1).mean() / EMBED_DIM).sqrt().item()
# (embed_dim,) true mean activation, the baseline for mean-ablation below
ACT_MEAN = sae_acts[:50_000].float().mean(0)
with torch.no_grad():
    sae.b_dec.copy_(sae_acts[:50_000].float().mean(0).to(DEVICE) / SCALE)
optimizer = torch.optim.Adam(sae.parameters(), lr=4e-4)
SAE_BATCH, SAE_EPOCHS = 2048, 3
fires = torch.zeros(sae.encoder.out_features, device=DEVICE)

start = time.time()
for epoch in range(SAE_EPOCHS):
    for i in range(0, len(sae_acts), SAE_BATCH):
        a = sae_acts[i : i + SAE_BATCH].float().to(DEVICE) / SCALE
        recon, z = sae(a)
        loss = (recon - a).pow(2).sum(-1).mean()
        optimizer.zero_grad()
        loss.backward()
        with torch.no_grad():
            # remove the gradient component along each decoder column
            w = sae.W_dec
            sae.W_dec.grad -= (w * sae.W_dec.grad).sum(0, keepdim=True) * w
        optimizer.step()
        sae.normalize_decoder()
        if epoch == SAE_EPOCHS - 1:
            fires += (z > 0).float().sum(0)
    fvu = loss.item() / (a - a.mean(0)).pow(2).sum(-1).mean().item()
    print(
        f"epoch {epoch}: loss {loss.item():.3f}, FVU {fvu:.3f} ({time.time() - start:.0f}s)"
    )

dead = (fires == 0).float().mean().item()
print(f"Dead latents (never fired in the last epoch): {dead:.1%}")

Reconstruction error only matters insofar as it changes the model's behaviour. The standard
downstream check splices the reconstruction $\hat a$ back into the forward pass in place of $a$ and
measures the increase in next-token loss on held-out text, compared with mean-ablating the layer
(replacing $a$ with the mean training activation).

In [ ]:
class SpliceHook:
    """Replace one layer's output with the SAE reconstruction or the mean activation."""

    def __init__(self, layer: nn.Module, mode: str) -> None:
        self.mode = mode
        self.handle = layer.register_forward_hook(self.hook)

    def hook(self, module, args, output):
        h = output[0] if isinstance(output, tuple) else output
        flat = h.reshape(-1, h.size(-1)).float()
        if self.mode == "sae":
            new = sae(flat / SCALE)[0] * SCALE
        else:
            new = ACT_MEAN.to(flat.device).expand_as(flat)
        new = new.view_as(h).to(h.dtype)
        # keep position 0 untouched: it was excluded from SAE training
        new[:, 0] = h[:, 0]
        return (new,) + tuple(output[1:]) if isinstance(output, tuple) else new


@torch.no_grad()
def lm_loss(texts: list[str]) -> float:
    """Mean next-token loss over texts."""
    losses = []
    for t in texts:
        ids = tokenizer(
            t, return_tensors="pt", truncation=True, max_length=CTX
        ).input_ids.to(DEVICE)
        losses.append(model(input_ids=ids, labels=ids).loss.item())
    return sum(losses) / len(losses)


held_out = web[-30:]
clean = lm_loss(held_out)
results = {"clean": clean}
for mode in ["sae", "mean"]:
    splice = SpliceHook(layers[BEST_LAYER], mode)
    results[mode] = lm_loss(held_out)
    splice.handle.remove()
print(
    f"LM loss clean {clean:.3f} | SAE spliced {results['sae']:.3f} "
    f"(+{results['sae'] - clean:.3f}) | mean-ablated {results['mean']:.3f} (+{results['mean'] - clean:.3f})"
)
print(
    f"Fraction of the layer's loss contribution recovered: "
    f"{(results['mean'] - results['sae']) / (results['mean'] - clean):.1%}"
)

## 4.7 Reading features and finding refusal without labels

Two ways to connect the dictionary to refusal:

- **Geometry.** Which decoder columns point along $\hat r$?
- **Activation statistics.** Which latents fire much more on harmful prompts than harmless ones at
  the last prompt token? This uses labels only to *find* features, not to fit them.

Each candidate is then read by its top-activating tokens on web text, the standard first step of
SAE interpretation.

In [ ]:
with torch.no_grad():
    cos = F.normalize(sae.W_dec, dim=0).T @ refusal_dir.to(DEVICE)
    pos_codes = sae.encode(test_pos[:, BEST_LAYER].to(DEVICE) / SCALE)
    neg_codes = sae.encode(test_neg[:, BEST_LAYER].to(DEVICE) / SCALE)
    freq_gap = (pos_codes > 0).float().mean(0) - (neg_codes > 0).float().mean(0)

top_geom = cos.topk(5).indices.tolist()
top_stat = freq_gap.topk(5).indices.tolist()
print("Most aligned with the refusal direction:")
for i in top_geom:
    print(
        f"  latent {i:>5}: cos {cos[i]:+.2f}, fires on harmful {(pos_codes[:, i] > 0).float().mean():.0%} / harmless {(neg_codes[:, i] > 0).float().mean():.0%}"
    )
print("Largest harmful-vs-harmless firing gap:")
for i in top_stat:
    print(
        f"  latent {i:>5}: gap {freq_gap[i]:+.0%}, cos with refusal dir {cos[i]:+.2f}"
    )

In [ ]:
@torch.no_grad()
def top_contexts(
    latent: int, texts: list[str], num: int = 6, width: int = 6
) -> list[str]:
    """Texts on which one latent activates most strongly, with the token where it peaks.

    For chat-formatted texts the user request is shown, since the peak is usually at the
    position where the model starts its reply.
    """
    hits = []
    for t in texts:
        ids = tokenizer(
            t, return_tensors="pt", truncation=True, max_length=CTX
        ).input_ids.to(DEVICE)
        h = model(input_ids=ids, output_hidden_states=True).hidden_states[
            BEST_LAYER + 1
        ][0, 1:]
        acts = sae.encode(h.float() / SCALE)[:, latent]
        best = acts.argmax().item()
        if "<|im_start|>user" in t:
            label = (
                "chat: " + t.split("<|im_start|>user\n")[1].split("<|im_end|>")[0][:70]
            )
        else:
            label = "web:  ..." + tokenizer.decode(
                ids[0, max(1, best - width) : best + 2]
            )
        hits.append((acts[best].item(), label))
    hits.sort(key=lambda x: -x[0])
    return [f"{v:5.1f} | {label!r}" for v, label in hits[:num] if v > 0]


probe_texts = (
    web[-120:-40]
    + chat(harmful_held[NUM_TEST:])
    + chat(harmless["test"]["text"][NUM_TEST : NUM_TEST + 40])
)
for latent in top_stat[:3]:
    print(f"latent {latent}:")
    for line in top_contexts(latent, probe_texts):
        print(f"   {line}")

## 4.8 Steering with an SAE feature

A feature's decoder column is a direction like any other. Adding the top harmful-vs-harmless
latent's column at `BEST_LAYER`, with a magnitude set to its typical activation on harmful
prompts, tests whether the unsupervised feature is causally tied to refusal.

In [ ]:
feature = top_stat[0]
feature_dir = sae.W_dec[:, feature].detach()
typical = pos_codes[:, feature][pos_codes[:, feature] > 0].mean().item() * SCALE
print(
    f"latent {feature}: cos with diff-in-means refusal dir {cos[feature]:+.2f}, typical activation {typical:.1f}"
)
for scale in [0.0, 5.0, 10.0, 20.0]:
    hook = ResidualHook(
        model, feature_dir, "add", alpha=scale * typical, layer_ids=[BEST_LAYER]
    )
    responses = generate(harmless_test[:16], max_new_tokens=40)
    hook.remove()
    print(
        f"{scale:.0f} x typical: refusal on harmless {refusal_rate(harmless_test[:16], responses):.0%} | {responses[0][:70]!r}"
    )

---
# 5) Results

| Intervention | Where | What changes |
|---|---|---|
| None | | Baseline refusal on held-out harmful and harmless prompts |
| Directional ablation (hooks) | All layers, all positions | Refusal on harmful drops; harmless behaviour almost unchanged (small first-token KL) |
| Weight orthogonalisation | Weights of every residual writer | Same logits as the hooked model; no inference overhead |
| Activation addition, diff-in-means | One layer | Harmless prompts start being refused as $\alpha$ grows |
| Activation addition, SAE latent | One layer | Tests whether an unlabelled feature carries the same causal role |

---
# 6) Practical notes

- **Layer choice is causal, not correlational.** Probe accuracy saturates early; the best layer for
  intervention is usually 40–65% of the depth, found by the ablation sweep.
- **Ablation vs addition.** Ablating everywhere is robust to the exact layer; adding at one layer
  needs a tuned $\alpha$, and too large an $\alpha$ breaks fluency before it changes behaviour.
- **Prompt-set quality.** Directions inherit any confound between the two sets (length, topic,
  formatting). Matched pairs (contrastive activation addition, Panickssery et al. 2024) reduce it.
- **Abliteration side effects.** Expect a small loss in benchmark accuracy; the projected variant
  removes only the component of $\hat r$ orthogonal to the harmless mean, which preserves more
  capability.
- **SAE scale.** Feature quality rises with dictionary size and training tokens; dead latents are
  controlled with TopK plus an auxiliary loss on the dead set (AuxK) and careful initialisation.
  Pretrained suites (Gemma Scope, Llama Scope) save the training cost.
- **Safety implication.** A refusal behaviour that a rank-1 edit removes is not a security
  boundary. Open-weight safety needs filtering of pretraining data, robust refusal training, and
  system-level guards (input/output classifiers like the guard model used here).

---
# 7) Conclusion

| Built from scratch | Takeaway |
|---|---|
| Residual capture and difference-in-means | One vector per concept from ~100 prompts per class |
| Midpoint linear probe | Shows at which depth a concept becomes linearly readable |
| Causal layer selection | Pick the layer whose ablation changes behaviour most at the lowest side-effect cost |
| Directional ablation and weight orthogonalisation | The same edit at inference time or baked into the weights |
| Activation addition | The direction is causal both ways: add it and harmless prompts are refused |
| TopK SAE | An unsupervised dictionary whose features can be read, matched to labelled directions and used for steering |